# Steane [[7,1,3]] Non-Holographic Control — IBM Fez Hardware

## Goal

Run the Steane [[7,1,3]] CSS code on IBM Fez as a **non-holographic control**
for the [[5,1,3]] HaPPY holographic experiment.

Both codes:
- Distance 3, correct any single-qubit error
- Run on the same chip (IBM Fez) with the same noise
- Analyzed with the same DBCI methodology

The difference: [[5,1,3]] has holographic (pentagon/tensor-network) structure.
Steane [[7,1,3]] is a CSS code with no holographic geometry.

## Key comparisons

1. **ΔH magnitude**: Both should show ΔH > 0 (same non-Pauli hardware noise)
2. **Isotropy (CV)**: Do stabilizer subsets give uniform or anisotropic ΔH?
3. **Complementary recovery**: Steane has CSS-type recovery but not holographic
4. **Decoding advantage**: DBCI vs Hard decoder on both codes

## Code

`[[7, 1, 3]]` Steane code. 6 stabilizers (3 X-type + 3 Z-type), 7 data qubits.
44 circuits (22 errors × 2 logical states), 8192 shots each.

In [1]:
"""Cell 1: Imports & Configuration"""

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "qiskit", "qiskit-aer", "qiskit-ibm-runtime",
                       "matplotlib", "numpy", "-q"])

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from itertools import combinations
import time

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Pauli, Statevector, Operator
from qiskit.synthesis import synth_circuit_from_stabilizers
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

SHOTS = 8192
SEED = 42
BACKEND_NAME = "ibm_fez"
np.random.seed(SEED)

print(f"Target: {BACKEND_NAME}, {SHOTS} shots/circuit")
print("Imports OK")

Target: ibm_fez, 8192 shots/circuit
Imports OK


In [2]:
"""Cell 2: Steane [[7,1,3]] Code Definition

The Steane code is a CSS code based on the classical [7,4,3] Hamming code.
6 stabilizers: 3 X-type (detect Z errors) + 3 Z-type (detect X errors).
"""

N_DATA = 7

# Stabilizers (left-to-right = qubit 0 to 6)
STABILIZERS = [
    "IIIXXXX",   # X on qubits 3,4,5,6
    "IXXIIXX",   # X on qubits 1,2,5,6
    "XIXIXIX",   # X on qubits 0,2,4,6
    "IIIZZZZ",   # Z on qubits 3,4,5,6
    "IZZIIZZ",   # Z on qubits 1,2,5,6
    "ZIZIZIZ",   # Z on qubits 0,2,4,6
]

N_ANC = len(STABILIZERS)

# Logical operators
Z_L = "ZZZZZZZ"
X_L = "XXXXXXX"

# Compute logical state vectors via stabilizer projection
from qiskit.quantum_info import Pauli

dim = 2**N_DATA

# Build projector onto codespace: product of (I + S_i)/2
proj = np.eye(dim, dtype=np.complex128)
for stab in STABILIZERS:
    P = Pauli(stab[::-1]).to_matrix()  # little-endian
    proj = proj @ ((np.eye(dim) + P) / 2)

# Project |0...0> into codespace
psi = proj @ np.eye(dim)[0]
norm = np.linalg.norm(psi)
print(f"Codespace projection norm: {norm:.6f}")

if norm < 1e-12:
    # Try all computational basis states
    for basis_idx in range(dim):
        psi = proj @ np.eye(dim)[basis_idx]
        norm = np.linalg.norm(psi)
        if norm > 1e-6:
            print(f"  Found nonzero projection at |{basis_idx:07b}>, norm={norm:.6f}")
            break

psi = psi / norm

# Split into Z_L eigenstates
Z_L_mat = Pauli(Z_L[::-1]).to_matrix()
z_vals = Z_L_mat @ psi  # Z_L|psi>

# |0_L> = (|psi> + Z_L|psi>)/2, |1_L> = (|psi> - Z_L|psi>)/2
v0 = (psi + z_vals) / 2
v1 = (psi - z_vals) / 2

n0 = np.linalg.norm(v0)
n1 = np.linalg.norm(v1)

if n0 > 1e-12:
    state0_vec = v0 / n0
else:
    state0_vec = v1 / n1
    v1 = v0
    n1 = n0

if n1 > 1e-12:
    state1_vec = v1 / n1
else:
    # Construct |1_L> = X_L |0_L>
    X_L_mat = Pauli(X_L[::-1]).to_matrix()
    state1_vec = X_L_mat @ state0_vec

from qiskit.quantum_info import Statevector
state0 = Statevector(state0_vec)
state1 = Statevector(state1_vec)

print(f"State vectors computed:")
print(f"  |0_L> norm: {np.linalg.norm(state0_vec):.6f}")
print(f"  |1_L> norm: {np.linalg.norm(state1_vec):.6f}")
print(f"  <0_L|1_L>:  {abs(state0_vec.conj() @ state1_vec):.6f}")

# Verify Z_L eigenvalues
z0 = state0_vec.conj() @ Z_L_mat @ state0_vec
z1 = state1_vec.conj() @ Z_L_mat @ state1_vec
print(f"  Z_L expectation: |0_L>={z0.real:+.3f}, |1_L>={z1.real:+.3f}")

# Error hypotheses: I + 21 single-qubit Paulis
ERRORS = ["I" * N_DATA]
LABELS = ["I"]
for q in range(N_DATA):
    for p in "XYZ":
        e = list("I" * N_DATA)
        e[q] = p
        ERRORS.append("".join(e))
        LABELS.append(f"{p}{q}")

N_HYP = len(ERRORS)

# Compute expected syndromes
def anticommutes(a, b):
    return not (a == "I" or b == "I" or a == b)

def syndrome_of(error_str):
    syn = []
    for stab in STABILIZERS:
        n = sum(anticommutes(e, s) for e, s in zip(error_str, stab))
        syn.append(n % 2)
    return tuple(syn)

EXPECTED_SYN = {label: syndrome_of(err) for label, err in zip(LABELS, ERRORS)}
unique_syns = len(set(EXPECTED_SYN.values()))

print(f"\nSteane [[7,1,3]]: {N_DATA} data + 1 ancilla = {N_DATA + 1} qubits")
print(f"  {N_HYP} error hypotheses, {unique_syns} unique syndromes")

syn_list = [syndrome_of(e) for e in ERRORS]
if len(set(syn_list)) == len(syn_list):
    print("  Non-degenerate: each error has a unique syndrome")
else:
    print("  WARNING: degenerate syndromes detected")
    from collections import Counter as C
    dup = [s for s, cnt in C(syn_list).items() if cnt > 1]
    for s in dup:
        matches = [l for l, syn in zip(LABELS, syn_list) if syn == s]
        print(f"    Syndrome {s}: {matches}")

Codespace projection norm: 0.353553
State vectors computed:
  |0_L> norm: 1.000000
  |1_L> norm: 1.000000
  <0_L|1_L>:  0.000000
  Z_L expectation: |0_L>=+1.000, |1_L>=-1.000

Steane [[7,1,3]]: 7 data + 1 ancilla = 8 qubits
  22 error hypotheses, 22 unique syndromes
  Non-degenerate: each error has a unique syndrome


In [3]:
"""Cell 3: Circuit Builder

Uses initialize() for state prep (same as [[5,1,3]] notebook).
Sequential ancilla reuse for syndrome extraction.

Note: a tiny non-Clifford perturbation (1e-10) is added to the state vector
to prevent the transpiler from recognizing and collapsing the Clifford circuit.
The physical effect is negligible (~1e-20 in fidelity).
"""

# Perturb state vectors to break Clifford recognition
state0_perturbed = state0_vec.copy()
state0_perturbed[0] *= np.exp(1j * 1e-10)
state0_perturbed /= np.linalg.norm(state0_perturbed)

state1_perturbed = state1_vec.copy()
state1_perturbed[0] *= np.exp(1j * 1e-10)
state1_perturbed /= np.linalg.norm(state1_perturbed)

print(f"Perturbation fidelity loss: {1 - abs(state0_vec.conj() @ state0_perturbed)**2:.2e}")

def build_circuit(logical_state, error_str):
    """Build: initialize -> error -> syndrome extraction -> measurement."""
    state_vec = state0_perturbed if logical_state == 0 else state1_perturbed

    qr = QuantumRegister(N_DATA, "data")
    anc = QuantumRegister(1, "anc")
    cr_syn = ClassicalRegister(N_ANC, "syn")
    cr_out = ClassicalRegister(N_DATA, "out")
    qc = QuantumCircuit(qr, anc, cr_syn, cr_out)

    # 1. State preparation
    qc.initialize(state_vec, qr)

    # 2. Error injection
    for q, p in enumerate(error_str):
        if p == "X": qc.x(qr[q])
        elif p == "Y": qc.y(qr[q])
        elif p == "Z": qc.z(qr[q])
    qc.barrier()

    # 3. Syndrome extraction (sequential ancilla reuse)
    for s_idx, stab in enumerate(STABILIZERS):
        qc.reset(anc[0])
        qc.h(anc[0])
        for q, pauli in enumerate(stab):
            if pauli == "X":
                qc.cx(anc[0], qr[q])
            elif pauli == "Z":
                qc.cz(anc[0], qr[q])
            elif pauli == "Y":
                qc.sdg(qr[q])
                qc.cx(anc[0], qr[q])
                qc.s(qr[q])
        qc.h(anc[0])
        qc.measure(anc[0], cr_syn[s_idx])
    qc.barrier()

    # 4. Measure data qubits
    qc.measure(qr, cr_out)

    return qc

# Build all 44 circuits
circuits = []
circuit_labels = []
for ls in [0, 1]:
    for err_str, err_label in zip(ERRORS, LABELS):
        qc = build_circuit(ls, err_str)
        circuits.append(qc)
        circuit_labels.append(f"L{ls}_{err_label}")

print(f"Built {len(circuits)} circuits")
print(f"Qubits: {circuits[0].num_qubits} ({N_DATA} data + 1 ancilla)")

Perturbation fidelity loss: -4.44e-16
Built 44 circuits
Qubits: 8 (7 data + 1 ancilla)


In [4]:
"""Cell 4: Noiseless Validation on Aer

Verify syndrome extraction before submitting to hardware.
"""

from qiskit_aer import AerSimulator

sim = AerSimulator()
t_circs = transpile(circuits, sim, optimization_level=1)
result_sim = sim.run(t_circs, shots=256, seed_simulator=SEED).result()

all_ok = True
for i, label in enumerate(circuit_labels):
    counts = result_sim.get_counts(i)
    syn_counts = Counter()
    for bitstring, count in counts.items():
        bits = bitstring.replace(" ", "")
        # With 7 data + 6 syn bits = 13 total
        # bitstring format: out(7) syn(6), but with sequential ancilla
        # the register layout is: out(7 bits) then syn(6 bits)
        syn_str = bits[N_DATA:]   # last N_ANC bits
        syn = tuple(int(b) for b in reversed(syn_str))
        syn_counts[syn] += count
    dominant = syn_counts.most_common(1)[0]
    err_label = label.split("_")[1]
    expected = EXPECTED_SYN[err_label]
    if dominant[0] != expected or dominant[1] < 250:
        print(f"FAIL: {label} expected {expected}, got {dominant[0]} ({dominant[1]}/256)")
        all_ok = False

if all_ok:
    print(f"All {len(circuits)} circuits: correct syndromes on Aer")
else:
    print("WARNING: fix syndrome extraction before submitting to hardware!")

All 44 circuits: correct syndromes on Aer


In [7]:
"""Cell 5: Connect to IBM Quantum & Transpile for Fez"""

IBM_TOKEN = "5GI3W65AZds2mLbWCaXyjr59CjZSFTBdfR8XJJsEywS8"
INSTANCE = "open-instance"
CHANNEL = "ibm_quantum_platform"

service = QiskitRuntimeService(
    channel=CHANNEL, token=IBM_TOKEN, instance=INSTANCE
)
backend = service.backend(BACKEND_NAME)
print(f"Connected to {backend.name}")
print(f"  Qubits: {backend.num_qubits}")

# Transpile for hardware
print(f"Transpiling {len(circuits)} circuits for {BACKEND_NAME}...")
t0 = time.time()
hw_circuits = transpile(
    circuits, backend=backend,
    optimization_level=3,
    seed_transpiler=SEED
)
elapsed = time.time() - t0

# Debug: show ALL gate types
ops = hw_circuits[0].count_ops()
print(f"\nAll gate types in circuit 0: {dict(ops)}")

depths = [qc.depth() for qc in hw_circuits]
all_2q = []
for qc in hw_circuits:
    ops = qc.count_ops()
    n2q = sum(v for k, v in ops.items() if k in ["cx", "ecr", "cz", "rzx", "rzz", "xx_plus_yy", "xx_minus_yy", "cp"])
    all_2q.append(n2q)

print(f"\nTranspiled in {elapsed:.1f}s")
print(f"  Depth: min={min(depths)}, max={max(depths)}, mean={np.mean(depths):.0f}")
print(f"  2Q gates: min={min(all_2q)}, max={max(all_2q)}, mean={np.mean(all_2q):.0f}")

if max(all_2q) == 0:
    print("WARNING: 0 two-qubit gates — do NOT submit to hardware!")
else:
    print("OK — circuits have entangling gates, safe to submit.")

qiskit_runtime_service._discover_account:WARNING:2026-03-07 11:30:07,123: Loading account with the given token. A saved account will not be used.


Connected to ibm_fez
  Qubits: 156
Transpiling 44 circuits for ibm_fez...

All gate types in circuit 0: {'sx': 468, 'rz': 341, 'cz': 272, 'x': 16, 'reset': 13, 'measure': 13, 'barrier': 2}

Transpiled in 10.5s
  Depth: min=612, max=617, mean=615
  2Q gates: min=272, max=272, mean=272
OK — circuits have entangling gates, safe to submit.


In [ ]:
"""Cell 6: Submit to IBM Fez

Uses SamplerV2. Single PUB with all circuits.
Set RESUME_JOB_ID to skip submission and retrieve a previous run.
"""

# IBM Fez, March 7 2026 — 44 circuits x 8192 shots
RESUME_JOB_ID = "d6m0ol0fh9oc73enfa1g"

if RESUME_JOB_ID is not None:
    print(f"Resuming from job {RESUME_JOB_ID}")
    job = service.job(RESUME_JOB_ID)
    result = job.result()
    print(f"Results retrieved ({len(result)} PUBs)")
else:
    sampler = Sampler(mode=backend)

    print(f"Submitting {len(hw_circuits)} circuits x {SHOTS} shots to {BACKEND_NAME}...")
    t0 = time.time()

    # Submit as individual PUBs (one per circuit)
    pubs = [(qc, [], SHOTS) for qc in hw_circuits]
    job = sampler.run(pubs)

    print(f"Job submitted: {job.job_id()}")
    print(f'RESUME_JOB_ID = "{job.job_id()}"')
    print(f"Waiting for results...")
    result = job.result()
    elapsed = time.time() - t0
    print(f"Results received in {elapsed:.0f}s")

In [9]:
"""Cell 7: Extract Syndrome & Data Arrays"""

def extract_bitstrings(pub_result, register_name):
    data = getattr(pub_result.data, register_name)
    bitstrings = data.get_bitstrings()
    arr = np.array([[int(b) for b in s[::-1]] for s in bitstrings], dtype=int)
    return arr

hw_data = {}
for i, label in enumerate(circuit_labels):
    pub_result = result[i]
    syn_arr = extract_bitstrings(pub_result, "syn")
    out_arr = extract_bitstrings(pub_result, "out")
    hw_data[label] = {"syndrome": syn_arr, "data": out_arr}

print(f"Extracted data for {len(hw_data)} circuits")
sample = hw_data["L0_I"]
syn_shape = sample["syndrome"].shape
dat_shape = sample["data"].shape
print(f"Syndrome shape: {syn_shape}")
print(f"Data shape: {dat_shape}")

# Syndrome accuracy for no-error circuits
for state in ["0", "1"]:
    syn = hw_data[f"L{state}_I"]["syndrome"]
    trivial = np.all(syn == 0, axis=1).mean()
    print(f"|{state}_L> no error: {trivial*100:.1f}% trivial syndrome")

Extracted data for 44 circuits
Syndrome shape: (8192, 6)
Data shape: (8192, 7)
|0_L> no error: 3.9% trivial syndrome
|1_L> no error: 3.4% trivial syndrome


In [ ]:
"""Cell 8: DeltaH Computation

Same methodology as [[5,1,3]] holographic experiment:
- Depolarizing prior (backward boundary)
- Empirical P(syndrome | error) from training half
- 2-fold cross-validation
- DeltaH = H[p_fwd] - H[p_DBCI]

HaPPY reference value computed from saved raw data (data/happy_553.npz).
"""

import os

def entropy_bits(p):
    p = np.asarray(p, dtype=np.float64)
    p = p[p > 0]
    return -np.sum(p * np.log2(p)) if len(p) > 0 else 0.0

def normalize_logits(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - np.max(logits)
    probs = np.exp(logits)
    Z = np.sum(probs)
    return probs / Z if Z > 1e-300 else np.ones_like(probs) / len(probs)

def compute_delta_H(hw_data_in, labels, n_hyp, n_anc, p_no_err=0.7,
                     stab_subset=None):
    """Compute DeltaH. If stab_subset given, use only those syndrome bits."""
    if stab_subset is None:
        stab_subset = list(range(n_anc))
    n_sub = len(stab_subset)
    n_syn_space = 2 ** n_sub

    log_prior_depol = np.zeros(n_hyp)
    log_prior_depol[0] = np.log(p_no_err)
    log_prior_depol[1:] = np.log((1 - p_no_err) / (n_hyp - 1))
    log_prior_uniform = np.zeros(n_hyp)
    powers = 2 ** np.arange(n_sub - 1, -1, -1)

    all_dH = []

    for state_label in ["0", "1"]:
        state_data = {}
        for err_idx, err_label in enumerate(labels):
            key = f"L{state_label}_{err_label}"
            if key in hw_data_in:
                state_data[err_idx] = hw_data_in[key]["syndrome"][:, stab_subset]

        if not state_data:
            continue

        n_shots = len(list(state_data.values())[0])
        mid = n_shots // 2

        for test_sl, train_sl in [(slice(0, mid), slice(mid, None)),
                                   (slice(mid, None), slice(0, mid))]:
            log_lik = np.full((n_hyp, n_syn_space), np.log(1.0 / n_syn_space))

            for h_idx in state_data:
                train_syn = state_data[h_idx][train_sl]
                keys = (train_syn * powers).sum(axis=1)
                counts = np.zeros(n_syn_space)
                for k in keys:
                    counts[k] += 1
                dist = (counts + 1) / (counts.sum() + n_syn_space)
                log_lik[h_idx] = np.log(dist)

            for err_idx in state_data:
                test_syn = state_data[err_idx][test_sl]
                syn_keys = (test_syn * powers).sum(axis=1)
                for syn_idx in syn_keys:
                    p_fwd = normalize_logits(log_lik[:, syn_idx] + log_prior_uniform)
                    p_dbci = normalize_logits(log_lik[:, syn_idx] + log_prior_depol)
                    all_dH.append(entropy_bits(p_fwd) - entropy_bits(p_dbci))

    return np.mean(all_dH), np.std(all_dH) / np.sqrt(len(all_dH))

# Full DeltaH — Steane
dH_full, sem_full = compute_delta_H(hw_data, LABELS, N_HYP, N_ANC)
print(f"Steane [[7,1,3]] DeltaH (full, {N_ANC}/{N_ANC}): {dH_full:.4f} +/- {sem_full:.4f} bits")

# Compute HaPPY reference from saved raw data
print("\nComputing [[5,1,3]] HaPPY reference from data/happy_553.npz...")
npz_path = os.path.join(os.path.dirname(os.getcwd()), "data", "happy_553.npz")
if not os.path.exists(npz_path):
    npz_path = os.path.join(os.getcwd(), "data", "happy_553.npz")
raw = np.load(npz_path)

# Build HaPPY labels and hw_data
happy_labels = ["I"]
for q in range(5):
    for p in "XYZ":
        happy_labels.append(f"{p}{q}")
happy_circuit_labels = [f"L{s}_{l}" for s in [0, 1] for l in happy_labels]
hw_happy_ref = {}
for i, label in enumerate(happy_circuit_labels):
    hw_happy_ref[label] = {
        "syndrome": raw[f"pub{i}_syn"].astype(int),
        "data":     raw[f"pub{i}_out"].astype(int),
    }

dH_happy_ref, sem_happy_ref = compute_delta_H(hw_happy_ref, happy_labels, 16, 4)
print(f"[[5,1,3]] HaPPY DeltaH: {dH_happy_ref:.4f} +/- {sem_happy_ref:.4f} bits")
print(f"Ratio: {dH_full / dH_happy_ref:.2f}x")

In [11]:
"""Cell 9: Isotropy Analysis -- CV of DeltaH across stabilizer subsets

Compute DeltaH for every C(6,k) subset of stabilizers.
Compare CV (coefficient of variation) with [[5,1,3]] HaPPY.
"""

results_steane = {}

print(f"Isotropy analysis: DeltaH per stabilizer subset")
print(f"{'k':>3} {'Subsets':>8} {'Mean dH':>10} {'CV':>8} {'Min':>10} {'Max':>10}")
print("-" * 55)

for k in range(1, N_ANC + 1):
    subs = list(combinations(range(N_ANC), k))
    dH_vals = []
    for sub in subs:
        dH, _ = compute_delta_H(hw_data, LABELS, N_HYP, N_ANC,
                                 stab_subset=list(sub))
        dH_vals.append(dH)
    mean_dH = np.mean(dH_vals)
    std_dH = np.std(dH_vals, ddof=0)
    cv = std_dH / mean_dH if mean_dH > 0 else 0
    results_steane[k] = {
        "mean": mean_dH, "std": std_dH, "cv": cv,
        "min": np.min(dH_vals), "max": np.max(dH_vals),
        "vals": dH_vals, "subsets": subs
    }
    print(f"{k:>3} {len(subs):>8} {mean_dH:>10.4f} {cv:>7.1%} {np.min(dH_vals):>10.4f} {np.max(dH_vals):>10.4f}")

# Compare with [[5,1,3]] at matched fraction
# [[5,1,3]] k=3/4: CV = 7.4%
# Steane k=5/6 (same ~83% fraction): CV = ?
print(f"\n--- Isotropy comparison (matched fraction ~83%) ---")
print(f"  [[5,1,3]] HaPPY  k=3/4: CV = 7.4%")
k_match = 5  # 5/6 = 83%
steane_cv = results_steane[k_match]["cv"]
print(f"  Steane [[7,1,3]] k={k_match}/{N_ANC}: CV = {steane_cv:.1%}")

Isotropy analysis: DeltaH per stabilizer subset
  k  Subsets    Mean dH       CV        Min        Max
-------------------------------------------------------
  1        6     2.2567    0.1%     2.2502     2.2599
  2       15     2.2522    0.2%     2.2448     2.2583
  3       20     2.2464    0.2%     2.2375     2.2532
  4       15     2.2386    0.2%     2.2303     2.2455
  5        6     2.2273    0.2%     2.2237     2.2342
  6        1     2.2104    0.0%     2.2104     2.2104

--- Isotropy comparison (matched fraction ~83%) ---
  [[5,1,3]] HaPPY  k=3/4: CV = 7.4%
  Steane [[7,1,3]] k=5/6: CV = 0.2%


In [12]:
"""Cell 10: DBCI vs Hard Decoding Comparison

Hard decoder: syndrome -> most likely error (LUT)
DBCI decoder: syndrome + empirical backward boundary
"""

# Build syndrome -> error LUT from theory
syn_to_error = {}
for err_str, err_label in zip(ERRORS, LABELS):
    syn = syndrome_of(err_str)
    syn_to_error[syn] = err_str

# Determine logical effect of each error
def logical_effect(error_str):
    """Does this error flip the logical qubit?"""
    n_anti = sum(anticommutes(e, z) for e, z in zip(error_str, Z_L))
    return n_anti % 2  # 1 = logical flip

# Hard decoder
hard_correct = 0
hard_total = 0
for label in circuit_labels:
    state = int(label[1])
    err_label = label.split("_")[1]
    err_str = ERRORS[LABELS.index(err_label)]
    true_flip = logical_effect(err_str)
    expected_out = state ^ true_flip  # what Z_L should read

    syn = hw_data[label]["syndrome"]
    out = hw_data[label]["data"]

    for shot in range(len(syn)):
        syn_tuple = tuple(syn[shot])
        if syn_tuple in syn_to_error:
            correction = syn_to_error[syn_tuple]
            corr_flip = logical_effect(correction)
        else:
            corr_flip = 0  # unknown syndrome -> no correction

        # Measure Z_L from data qubits
        z_l_parity = sum(out[shot]) % 2
        decoded = z_l_parity ^ corr_flip

        if decoded == expected_out:
            hard_correct += 1
        hard_total += 1

hard_fidelity = hard_correct / hard_total
hard_error = 1 - hard_fidelity

# DBCI decoder (using empirical likelihoods from training half)
dbci_correct = 0
dbci_total = 0

n_syn_space = 2 ** N_ANC
log_prior = np.zeros(N_HYP)
log_prior[0] = np.log(0.7)
log_prior[1:] = np.log(0.3 / (N_HYP - 1))
powers = 2 ** np.arange(N_ANC - 1, -1, -1)

for state_label in ["0", "1"]:
    state = int(state_label)
    # Build empirical likelihoods from training half
    log_lik = np.full((N_HYP, n_syn_space), np.log(1.0 / n_syn_space))
    for err_idx, err_label in enumerate(LABELS):
        key = f"L{state_label}_{err_label}"
        train_syn = hw_data[key]["syndrome"][SHOTS//2:]
        syn_keys = (train_syn * powers).sum(axis=1)
        counts = np.zeros(n_syn_space)
        for k in syn_keys:
            counts[k] += 1
        dist = (counts + 1) / (counts.sum() + n_syn_space)
        log_lik[err_idx] = np.log(dist)

    # Decode test half
    for err_idx, err_label in enumerate(LABELS):
        key = f"L{state_label}_{err_label}"
        true_flip = logical_effect(ERRORS[err_idx])
        expected_out = state ^ true_flip

        test_syn = hw_data[key]["syndrome"][:SHOTS//2]
        test_out = hw_data[key]["data"][:SHOTS//2]
        syn_keys = (test_syn * powers).sum(axis=1)

        for shot in range(len(test_syn)):
            posterior = normalize_logits(log_lik[:, syn_keys[shot]] + log_prior)
            best_h = np.argmax(posterior)
            corr_flip = logical_effect(ERRORS[best_h])
            z_l_parity = sum(test_out[shot]) % 2
            decoded = z_l_parity ^ corr_flip
            if decoded == expected_out:
                dbci_correct += 1
            dbci_total += 1

dbci_fidelity = dbci_correct / dbci_total
dbci_error = 1 - dbci_fidelity

print(f"=== Decoding Results ===")
print(f"  Hard (LUT):  {hard_fidelity*100:.2f}% fidelity ({hard_error*100:.2f}% error)")
print(f"  DBCI:        {dbci_fidelity*100:.2f}% fidelity ({dbci_error*100:.2f}% error)")
if hard_error > 0:
    print(f"  Advantage:   {hard_error/dbci_error:.1f}x error reduction")
print(f"\n[[5,1,3]] HaPPY reference: Hard 10.49%, DBCI 31.20% single-shot fidelity")

=== Decoding Results ===
  Hard (LUT):  50.29% fidelity (49.71% error)
  DBCI:        50.50% fidelity (49.50% error)
  Advantage:   1.0x error reduction

[[5,1,3]] HaPPY reference: Hard 10.49%, DBCI 31.20% single-shot fidelity


In [ ]:
"""Cell 11: Summary Figure"""

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) DeltaH per stabilizer count
ax = axes[0]
ks = list(results_steane.keys())
means = [results_steane[k]["mean"] for k in ks]
stds = [results_steane[k]["std"] for k in ks]
ax.errorbar(ks, means, yerr=stds, fmt="o-", color="darkorange", lw=2, ms=8,
            capsize=4, label="Steane [[7,1,3]]", zorder=4)
ax.axhline(y=dH_happy_ref, color="steelblue", ls="--", lw=2,
           label=f"[[5,1,3]] HaPPY (full, {dH_happy_ref:.4f})", alpha=0.8)
ax.set_xlabel("Stabilizers used (k)", fontsize=12)
ax.set_ylabel("\u0394H (bits)", fontsize=12)
ax.set_title("(a) \u0394H vs Stabilizer Count", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# (b) CV comparison
ax = axes[1]
cvs = [results_steane[k]["cv"] * 100 for k in ks if k < N_ANC]
ks_partial = [k for k in ks if k < N_ANC]
ax.bar(ks_partial, cvs, color="darkorange", alpha=0.7, label="Steane [[7,1,3]]")
# [[5,1,3]] reference at matched fractions
happy_cvs = {1: 6.6, 2: 8.2, 3: 7.4}  # from partial-boundary notebook
for k_h, cv_h in happy_cvs.items():
    frac = k_h / 4
    k_s = round(frac * N_ANC)
    if k_s in results_steane and k_s < N_ANC:
        ax.scatter(k_s, cv_h, marker="*", s=200, color="steelblue", zorder=5,
                   label="[[5,1,3]] HaPPY (matched frac)" if k_h == 1 else "")
ax.set_xlabel("Stabilizers used (k)", fontsize=12)
ax.set_ylabel("CV of \u0394H across subsets (%)", fontsize=12)
ax.set_title("(b) Isotropy: CV Comparison", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("steane_vs_happy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: steane_vs_happy_comparison.png")